<h1 style="text-align: center;"> Data Preprocessing & Feature Engineering </h1>
<h4 style="text-align: center;">  From Raw Data to Model-Ready Features </h4>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns
import plotly.express as px

# Set Float precision
pd.options.display.float_format = '{:,.3f}'.format
# Set visual style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
df.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Brand,Model
0,Mumbai,2010,72000,CNG,Manual,First,11.438,998.000,58.160,5.000,1.750,Maruti,Wagon
1,Pune,2015,41000,Diesel,Manual,First,19.670,"1,582.000",126.200,5.000,12.500,Hyundai,Creta
2,Chennai,2011,46000,Petrol,Manual,First,18.200,"1,199.000",88.700,5.000,4.500,Honda,Jazz
3,Chennai,2012,87000,Diesel,Manual,First,20.770,"1,248.000",88.760,7.000,6.000,Maruti,Ertiga
4,Coimbatore,2013,40670,Diesel,Automatic,Second,15.200,"1,968.000",140.800,5.000,17.740,Audi,A4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6015 entries, 0 to 6014
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           6015 non-null   object 
 1   Year               6015 non-null   int64  
 2   Kilometers_Driven  6015 non-null   int64  
 3   Fuel_Type          6015 non-null   object 
 4   Transmission       6015 non-null   object 
 5   Owner_Type         6015 non-null   object 
 6   Mileage            5945 non-null   float64
 7   Engine             5979 non-null   float64
 8   Power              5873 non-null   float64
 9   Seats              5973 non-null   float64
 10  Price              6015 non-null   float64
 11  Brand              6015 non-null   object 
 12  Model              6015 non-null   object 
dtypes: float64(5), int64(2), object(6)
memory usage: 611.0+ KB


In [4]:
df['Seats'] = df['Seats'].astype('category')

In [5]:
df.describe()

,Year,Kilometers_Driven,Mileage,Engine,Power,Price
count,"6,015.000","6,015.000","5,945.000","5,979.000","5,873.000","6,015.000"
mean,"2,013.358","57,670.792",18.195,"1,619.955",113.128,9.425
std,3.270,"37,870.190",4.152,598.896,53.507,10.905
min,"1,998.000",171.000,5.676,72.000,34.200,0.440
25%,"2,011.000","34,000.000",15.150,"1,198.000",75.000,3.500
50%,"2,014.000","53,000.000",18.120,"1,493.000",97.700,5.630
75%,"2,016.000","73,000.000",21.030,"1,984.000",138.100,9.950
max,"2,019.000","775,000.000",28.400,"5,998.000",552.000,100.000


In [6]:
# First, Split the data into train and test sets
from sklearn.model_selection import train_test_split

X = df.drop(['Price'] , axis=1)
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
num_features = X.select_dtypes(include=[np.number]).columns.to_list()
num_features

['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Power']

# Numerical Features Transformations

### ✅ Log Transformation

**Purpose:**
- Reduce **positive skew** in data (long tail on the right).  
- Makes distributions closer to **normal**, useful for linear models.  
- Reduces the **impact of large outliers**.

**Mathematical Definition:**

For a positive feature $x > 0$:

$$
x_{\text{log}} = \log(x)
$$

To handle zeros safely, use:

$$
x_{\text{log1p}} = \log(1 + x)
$$



In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

# Sample data
df = pd.DataFrame({'salary': [30000, 50000, 70000, 100000, 250000]})

# Define log transformer (handles positive values)
log_transformer = FunctionTransformer(np.log1p)  # log1p(x) = log(1 + x)

# Apply transformation
df['salary_log'] = log_transformer.fit_transform(df[['salary']])
df


,salary,salary_log
0,30000,10.309
1,50000,10.820
2,70000,11.156
3,100000,11.513
4,250000,12.429


### ✅ Power Transformation / Box-Cox

**Purpose:**
- Reduce skewness and **stabilize variance**.  
- Makes data more **normal-like**, improving model performance.  
- Can be applied as **Box-Cox** (positives only) or **Yeo-Johnson** (allows 0 or negative).




In [9]:
from sklearn.preprocessing import PowerTransformer

# Box-Cox transformation (positive values only)
pt_boxcox = PowerTransformer(method='box-cox')
df['salary_boxcox'] = pt_boxcox.fit_transform(df[['salary']])

# Yeo-Johnson transformation (allows 0 or negative)
pt_yeojohnson = PowerTransformer(method='yeo-johnson')
df['salary_yeojohnson'] = pt_yeojohnson.fit_transform(df[['salary']])

df


,salary,salary_log,salary_boxcox,salary_yeojohnson
0,30000,10.309,-1.462,-1.462
1,50000,10.820,-0.540,-0.540
2,70000,11.156,-0.012,-0.012
3,100000,11.513,0.487,0.487
4,250000,12.429,1.528,1.528


### ✅ Binning / Discretization

**Purpose:**
- Convert **continuous numeric variables** into **categories**.  
- Reduce noise, capture **non-linear effects**, or create **ordinal features**.


**Key Notes:**
- `strategy='uniform'`: equal width bins.  
- `strategy='quantile'`: equal number of samples per bin.  
- `strategy='kmeans'`: bins based on clustering.  
- Choose the number of bins carefully: 
    - too many → overfitting, 
    - too few → loss of information.


In [10]:
import pandas as pd

df = pd.DataFrame({'age': [18, 22, 25, 30, 35, 40, 50]})

# Define bins manually
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 50], labels=['Young', 'Adult', 'Senior'])
df


,age,age_group
0,18,Young
1,22,Young
2,25,Young
3,30,Adult
4,35,Adult
5,40,Senior
6,50,Senior


In [11]:
from sklearn.preprocessing import KBinsDiscretizer

X = df[['age']]
kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')  # numeric output
df['age_binned'] = kbd.fit_transform(X)
df


,age,age_group,age_binned
0,18,Young,0.000
1,22,Young,0.000
2,25,Young,0.000
3,30,Adult,1.000
4,35,Adult,1.000
5,40,Senior,2.000
6,50,Senior,2.000


In [12]:
from sklearn.preprocessing import KBinsDiscretizer

# Sample data
df = pd.DataFrame({'age': [18, 22, 25, 30, 35, 40, 50]})

# KBinsDiscretizer: convert continuous 'age' into 3 bins
kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')  # uniform width bins
df['age_binned_uniform'] = kbd.fit_transform(df[['age']])

# Quantile bins (equal number of samples per bin)
kbd_quantile = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
df['age_binned_quantile'] = kbd_quantile.fit_transform(df[['age']])

df


,age,age_binned_uniform,age_binned_quantile
0,18,0.000,0.000
1,22,0.000,0.000
2,25,0.000,1.000
3,30,1.000,1.000
4,35,1.000,2.000
5,40,2.000,2.000
6,50,2.000,2.000


In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Suppose num_features is a list of numeric column names
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scaler', MinMaxScaler())
])

transformer = ColumnTransformer([
    ('num', num_pipeline, num_features)
], remainder='drop').set_output(transform='pandas')


In [14]:
X_train_prepared = transformer.fit_transform(X_train)
X_test_prepared = transformer.transform(X_test)

X_train_prepared

,num__Year,num__Kilometers_Driven,num__Mileage,num__Engine,num__Power
5639,0.382,0.809,0.583,0.699,0.360
2303,0.382,0.767,0.646,0.685,0.383
2617,0.477,0.787,0.728,0.669,0.250
1397,0.858,0.634,0.608,0.842,0.725
3703,0.715,0.677,0.728,0.698,0.467
...,...,...,...,...,...
3772,0.810,0.531,0.721,0.635,0.311
5191,0.763,0.664,0.764,0.685,0.340
5226,0.668,0.719,0.846,0.750,0.614
5390,0.573,0.662,0.721,0.635,0.311


In [15]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, method='iqr', lower_percentile=25, upper_percentile=75,
                 iqr_factor=1.5, z_thresh=3.0, handle='winsorize'):
        self.method = method
        self.lower_percentile = lower_percentile
        self.upper_percentile = upper_percentile
        self.iqr_factor = iqr_factor
        self.z_thresh = z_thresh
        self.handle = handle
        self.bounds_ = {}
        self.columns_ = None

    def fit(self, X, y=None):
        # Ensure input is a DataFrame
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)

        self.columns_ = X.select_dtypes(include=np.number).columns

        if self.method == 'iqr':
            Q1 = X[self.columns_].quantile(self.lower_percentile / 100.0)
            Q3 = X[self.columns_].quantile(self.upper_percentile / 100.0)
            IQR = Q3 - Q1
            lower = Q1 - self.iqr_factor * IQR
            upper = Q3 + self.iqr_factor * IQR
        elif self.method == 'zscore':
            mean = X[self.columns_].mean()
            std = X[self.columns_].std()
            lower = mean - self.z_thresh * std
            upper = mean + self.z_thresh * std
        else:
            raise ValueError("method must be 'iqr' or 'zscore'")

        self.bounds_ = {col: (lower[col], upper[col]) for col in self.columns_}
        return self

    def transform(self, X):
        # Convert to DataFrame (keep column names if possible)
        if isinstance(X, np.ndarray):
            if self.columns_ is not None and X.shape[1] == len(self.columns_):
                X = pd.DataFrame(X, columns=self.columns_)
            else:
                X = pd.DataFrame(X)

        X_copy = X.copy()

        # Apply outlier handling
        for col, (lower, upper) in self.bounds_.items():
            if self.handle == 'remove':
                X_copy[col] = X_copy[col].where((X_copy[col] >= lower) & (X_copy[col] <= upper))
            elif self.handle == 'winsorize':
                X_copy[col] = np.clip(X_copy[col], lower, upper)
            else:
                raise ValueError("handle must be 'remove' or 'winsorize'")
        
        return X_copy

    def fit_transform(self, X, y=None, **fit_params):
        return self.fit(X, y).transform(X)

    def get_feature_names_out(self, input_features=None):
        # Return the same feature names
        if input_features is None and hasattr(self, "columns_"):
            return self.columns_
        return input_features


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

num_features = ['Mileage', 'Engine', 'Power']
year_cols = ['Year']
log_cols = ['Kilometers_Driven']

# Pipeline for log column
log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p)),
    ('scaler', MinMaxScaler())
])

# Pipeline for year column
year_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

# Pipeline for remaining numeric columns
other_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier_handler', OutlierHandler(method='iqr', handle='winsorize')),
    ('scaler', MinMaxScaler())
])

# Combine into ColumnTransformer
preprocessor = ColumnTransformer([
    ('log_num', log_pipeline, log_cols),
    ('year', year_pipeline, year_cols),
    ('other_num', other_pipeline, num_features)
], verbose_feature_names_out=False).set_output(transform='pandas')

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

In [17]:
X_train_prepared

,Kilometers_Driven,Year,Mileage,Engine,Power
5639,0.809,0.381,0.399,0.497,0.308
2303,0.767,0.381,0.468,0.467,0.339
2617,0.787,0.476,0.570,0.434,0.179
1397,0.634,0.857,0.425,0.957,1.000
3703,0.677,0.714,0.569,0.495,0.475
...,...,...,...,...,...
3772,0.531,0.810,0.561,0.369,0.247
5191,0.664,0.762,0.618,0.467,0.282
5226,0.719,0.667,0.736,0.630,0.804
5390,0.662,0.571,0.561,0.369,0.246


**Takeaway:**

- If your data has **extreme outliers** that distort scaling or modeling, `OutlierHandler` (especially with **winsorization**) is often **more controlled and robust** than a log or power transform.
- For **moderate skew**, log or power transforms can help **normalize the distribution**.
- You can use **a combined approach**:
  1. **Handle extreme outliers** using `OutlierHandler`
  2. **Transform skewed features** with `PowerTransformer` or log
  3. **Scale features** using `MinMaxScaler` or `StandardScaler`


### ✅ Polynomial Features

**Purpose:**
- Capture **non-linear relationships** using linear models.
- Expand the feature set by creating **combinations of powers and interactions**.

**Mathematical Definition:**

Given features $x_1, x_2, ..., x_n$ and polynomial degree $d$, the expanded feature set includes all terms:

$$
1, x_1, x_2, ..., x_1^2, x_1 x_2, x_2^2, ..., x_1^d, x_1^{d-1} x_2, ..., x_n^d
$$

- **Degree $d$**: maximum power of polynomial terms.
- **Interaction-only**: include only multiplicative combinations like $x_i x_j$.
- **Bias term**: optional column of 1s for intercept.


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  PolynomialFeatures, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


# ---- Pipeline for polynomial columns ----
poly_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier', OutlierHandler(method='iqr', handle='winsorize')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', MinMaxScaler())
])


preprocessor = ColumnTransformer([
    ('poly', poly_pipeline, num_features),
]).set_output(transform='pandas')

# ---- Fit and transform ----
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)


In [19]:
X_train_prepared

,poly__Mileage,poly__Engine,poly__Power,poly__Mileage^2,poly__Mileage Engine,poly__Mileage Power,poly__Engine^2,poly__Engine Power,poly__Power^2
5639,0.399,0.497,0.308,0.252,0.417,0.214,0.259,0.206,0.151
2303,0.468,0.467,0.339,0.315,0.431,0.266,0.229,0.207,0.174
2617,0.570,0.434,0.179,0.419,0.456,0.186,0.199,0.132,0.071
1397,0.425,0.957,1.000,0.275,0.836,0.682,0.918,0.958,1.000
3703,0.569,0.495,0.475,0.418,0.519,0.419,0.256,0.277,0.291
...,...,...,...,...,...,...,...,...,...
3772,0.561,0.369,0.247,0.409,0.383,0.236,0.146,0.134,0.109
5191,0.618,0.467,0.282,0.473,0.519,0.287,0.229,0.183,0.132
5226,0.736,0.630,0.804,0.617,0.790,0.824,0.407,0.530,0.687
5390,0.561,0.369,0.246,0.409,0.383,0.235,0.146,0.134,0.109


In [20]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer

# Define log transformer (handles positive values)
log_transformer = FunctionTransformer(np.log1p)  # log1p(x) = log(1 + x)

# Apply transformation on car prices
y_train_log = log_transformer.fit_transform(y_train.to_frame())
y_test_log = log_transformer.transform(y_test.to_frame())



In [23]:
px.histogram(y_train, nbins=50, title='Log-Transformed Car Prices Distribution', labels={'value': 'Price '}).show()

In [21]:
px.histogram(y_train_log, nbins=50, title='Log-Transformed Car Prices Distribution', labels={'value': 'Log(Price + 1)'}).show()